# Cours — Réseaux de neurones récurrents (RNN)

**Semaine 5 · Jour 4** · Propilote-Learning

**Au programme**

1. Introduction aux réseaux de neurones récurrents (RNN)
2. Comprendre les LSTM et le problème du gradient évanescent
3. Le gradient explosif et les unités récurrentes à portes (GRU)
4. Tutoriel : prédire un cours de bourse avec LSTM et GRU

> **Comment utiliser ce notebook**
>
> 1. **Fichier → Enregistrer une copie dans Drive** pour travailler sur ta propre copie.
> 2. Lis chaque leçon et **exécute les cellules de code** (Maj + Entrée) : modifie-les, teste, observe.
> 3. Les lignes `# → …` indiquent le résultat attendu.
> 4. Les cellules **✍️ Essaie ici** sont pour toi : fais les mini-exercices du cours.
> 5. Certaines cellules montrent **volontairement une erreur** (le texte autour l'explique) : lis le message, c'est le but.
> 6. Les exercices notés de la journée sont dans les notebooks d'exercices (liens en bas).

# 1. Introduction aux réseaux de neurones récurrents (RNN)

*Architecture, traitement des séquences, applications et limites*

**Objectifs**

- Comprendre l'architecture de base d'un RNN
- Voir comment un RNN traite des données séquentielles
- Connaître les applications aux séries temporelles et au texte
- Identifier les limites des RNN classiques

## Qu'est-ce qu'un RNN ?

Un réseau de neurones récurrent (RNN) est conçu pour les données séquentielles. Contrairement à un réseau classique qui traite chaque entrée indépendamment, ses connexions forment des boucles : il garde une « mémoire » des entrées précédentes dans un état caché. Idéal quand l'ordre et le moment des données comptent.

- prévision de séries temporelles (cours de bourse) ;
- traitement du langage (génération de texte, traduction) ;
- reconnaissance vocale ;
- analyse vidéo.

Exemple de prévision boursière : à chaque instant t, le RNN reçoit le prix du moment, met à jour son état caché avec cette entrée et l'état précédent, puis prédit le prix de l'instant t+1.

## ANN, CNN, RNN

| Critère | ANN | CNN | RNN |
| --- | --- | --- | --- |
| Usage | Données tabulaires, classification ou régression simple | Images et données spatiales | Séquences : séries temporelles, texte, parole |
| Architecture | Couches entièrement connectées | Convolutions, pooling, couches denses | Couches récurrentes avec boucles |
| Données | Points indépendants | Données spatialement corrélées | Données dépendantes dans le temps |
| Mémoire | Aucune | Aucune (caractéristiques locales) | État caché qui retient le passé |
| Partage des poids | Non | Dans les convolutions | Entre les pas de temps |
| Rétropropagation | Classique | À travers les convolutions | Dans le temps (BPTT) |
| Exemples | Prix immobiliers, fraude | ResNet, VGG, AlexNet | Génération de texte, traduction |

> ✏️ **À toi**
>
> Tu analyses des conversations du support client pour repérer des motifs dans les plaintes. Quel type de réseau utiliser, et pourquoi ?

In [ ]:
# ✍️ Essaie ici


## Architecture

- Couche d'entrée : reçoit la séquence un élément à la fois.
- Couche cachée (récurrente) : traite l'entrée et met à jour l'état caché, qui sert de mémoire.
- Couche de sortie : produit une prédiction à partir de l'entrée courante et de l'état caché.

```python
h_t = tanh(W_hh · h_(t-1) + W_xh · x_t + b_h)
y_t = W_hy · h_t + b_y
```

Les mêmes poids (W_hh, W_xh, W_hy) servent à chaque pas de temps. Ce partage des paramètres permet de traiter des séquences de longueur variable, réduit le nombre de paramètres et aide à capter les dépendances.

> ✏️ **À toi**
>
> Un assistant doit répondre en tenant compte des messages précédents de l'utilisateur. Quelle affirmation explique le mieux pourquoi un RNN convient ?
>
> - A) Les RNN traitent chaque entrée indépendamment, sans tenir compte du passé.
> - B) Les RNN gardent un état caché qui mémorise les entrées précédentes et leur permet de comprendre le contexte.
> - C) Les RNN sont conçus pour la reconnaissance d'images.

In [ ]:
# ✍️ Essaie ici


## Applications

L'analyse de séries temporelles étudie des données relevées à intervalles réguliers, en cherchant tendances et motifs dans le temps (et non des observations indépendantes).

| Application | Exemple |
| --- | --- |
| Prévision boursière | Prédire le prochain cours à partir de l'historique |
| Météo | Prévoir la température à partir de la température, de l'humidité et du vent passés |
| Détection d'anomalies | Repérer un comportement inhabituel dans des journaux serveur (cyberattaque) ou des transactions |
| Texte et parole | Prédire le mot suivant, transcrire, traduire |

> ✏️ **À toi**
>
> Une équipe de météorologues veut prédire la température de demain. Comment un RNN s'appuie-t-il sur les conditions passées ? Quel est le rôle de l'état caché ?

In [ ]:
# ✍️ Essaie ici


## Limites

| Limite | Cause | Exemple |
| --- | --- | --- |
| Gradient évanescent | Pendant la rétropropagation dans le temps, les gradients rétrécissent de façon exponentielle (sigmoïde, tanh) : le réseau n'apprend que le court terme | Oublier le début d'une longue phrase au moment de prédire le dernier mot |
| Gradient explosif | Les gradients grossissent de façon exponentielle : mises à jour énormes, entraînement instable | Surréagir aux petites variations d'un cours de bourse |
| Coût de calcul | Chaque pas dépend du précédent : pas de parallélisation, mémoire importante, entraînement long | Reconnaissance vocale sur des heures d'audio |

Les leçons suivantes présentent les solutions : LSTM, GRU et écrêtage du gradient.

> ✏️ **À toi**
>
> Ton chatbot oublie ce qui a été dit plus tôt dans la conversation. Quelle limite des RNN en est probablement la cause, et comment y remédier ?

In [ ]:
# ✍️ Essaie ici


# 2. Comprendre les LSTM et le problème du gradient évanescent

*Cellule mémoire, portes d'entrée, d'oubli et de sortie*

**Objectifs**

- Comprendre le problème du gradient évanescent
- Connaître l'architecture d'un LSTM
- Expliquer le rôle de la cellule mémoire et des trois portes
- Connaître les applications et les limites des LSTM

## Le gradient évanescent

Un RNN apprend en ajustant ses poids grâce à la rétropropagation de l'erreur. Mais dans un RNN classique, le signal d'erreur (le gradient) rétrécit à chaque pas en remontant le temps : arrivé au début de la séquence, il est si faible qu'il ne modifie presque plus les poids. Le réseau n'arrive pas à relier des événements éloignés.

C'est comme retenir une longue histoire : si elle est courte, on relie facilement le début à la fin ; si elle est très longue, on a oublié le début en arrivant à la fin.

Exemple : « The dog, which was very playful, chased the ball. » Pour prédire « chased », il faut se souvenir du sujet « the dog », dilué par « which was very playful ».

- Les dépendances à long terme sont ignorées : le réseau privilégie les entrées récentes.
- L'entraînement devient inefficace : les poids évoluent très lentement.
- Les performances chutent sur les tâches réelles (traduction, prévision boursière).

> ✏️ **À toi**
>
> Pour un modèle qui résume de longs articles, comment le gradient évanescent peut-il nuire à la qualité des résumés ? Quelles techniques permettent de l'atténuer ?

In [ ]:
# ✍️ Essaie ici


## L'architecture LSTM

Les LSTM (Long Short-Term Memory) sont des RNN conçus pour contourner le gradient évanescent. Leur innovation : une cellule mémoire et des portes qui décident quoi retenir ou oublier, sur de longues séquences.

| Composant | Rôle |
| --- | --- |
| Cellule mémoire | Stocke l'information dans la durée sans la dégrader |
| Porte d'entrée | Décide quelle nouvelle information ajouter à la cellule |
| Porte d'oubli | Décide quelle information effacer de la cellule |
| Porte de sortie | Décide quelle partie de la cellule sert à la sortie et au prochain état caché |

Chaque porte est un petit réseau dont la sortie est comprise entre 0 (rien ne passe) et 1 (tout passe).

## Un LSTM pas à pas

- 1. Le LSTM reçoit l'entrée courante et l'état caché précédent.
- 2. Il calcule les valeurs des trois portes.
- 3. Il met à jour la cellule mémoire : la porte d'oubli retire l'inutile, la porte d'entrée ajoute le nouveau.
- 4. La porte de sortie détermine la sortie du pas courant et le nouvel état caché.

Exemple avec « The cat, which was very hungry, ate the food. » :

| Pas | Mot traité | Ce qui se passe |
| --- | --- | --- |
| 1-2 | The cat | La cellule mémorise le sujet ; la porte d'oubli le conserve |
| 3 | which was very hungry | La porte d'entrée ajoute le contexte « affamé » ; le sujet est conservé |
| 4 | ate | La porte de sortie rappelle le sujet et le contexte pour prédire la suite |

> ✏️ **À toi**
>
> Dans un système de traduction en temps réel de longues phrases, comment la cellule mémoire et les portes d'oubli, d'entrée et de sortie maintiennent-elles le contexte ?

In [ ]:
# ✍️ Essaie ici


## Applications et limites

| Domaine | Exemples |
| --- | --- |
| Séries temporelles | Cours de bourse, météo, consommation d'énergie |
| Langage | Génération de texte, traduction, analyse de sentiment, reconnaissance vocale |
| Autres | Analyse vidéo, génération musicale, détection d'anomalies |

| Limite | Détail |
| --- | --- |
| Coût de calcul | Plus lourd qu'un RNN classique |
| Hyperparamètres | Couches, unités, taux d'apprentissage à régler |
| Surapprentissage | Sur de petits jeux : dropout ou weight decay nécessaires |
| Séquences très longues | Les mécanismes d'attention et les Transformers font mieux |
| Interprétabilité | Boîte noire difficile à expliquer |
| Mémoire | États et gradients à stocker pendant l'entraînement |

# 3. Le gradient explosif et les unités récurrentes à portes (GRU)

*Écrêtage du gradient, architecture GRU et comparaison RNN / LSTM / GRU*

**Objectifs**

- Comprendre le gradient explosif et ses effets
- Le corriger par écrêtage du gradient et bonne initialisation
- Connaître l'architecture des GRU
- Choisir entre RNN, LSTM et GRU

## Le gradient explosif

Pendant l'entraînement, le gradient est multiplié par la même matrice de poids à chaque pas de temps. Si ces poids sont grands (> 1), il grossit de façon exponentielle, comme une boule de neige qui dévale une pente raide : les mises à jour deviennent énormes et l'entraînement instable.

- Entraînement instable : le modèle ne converge pas.
- Instabilité numérique : dépassement de capacité, l'entraînement peut planter.
- Prédictions erratiques : par exemple des cours de bourse prédits qui oscillent brutalement.

| Solution | Principe |
| --- | --- |
| Écrêtage du gradient | Si la norme du gradient dépasse un seuil, on la ramène à ce seuil (la plus courante) |
| Initialisation des poids | Xavier ou He : des poids ni trop grands ni trop petits au départ |
| Architectures à portes | LSTM et GRU contrôlent mieux le flux d'information |

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):                 # x : (batch, seq_len, input_size)
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])     # dernier pas de temps

model = SimpleRNN(10, 20, 1)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

x, y = torch.randn(16, 5, 10), torch.randn(16, 1)
optimizer.zero_grad()
loss = criterion(model(x), y)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)   # écrêtage
optimizer.step()
print(f"Loss after one step: {loss.item():.4f}")

Juste après backward(), clip_grad_norm_ limite la norme totale des gradients à 5, avant que l'optimiseur ne mette à jour les poids.

> ✏️ **À toi**
>
> Ton RNN de prévision boursière produit des prédictions qui varient brutalement. Quelle en est la cause probable ? Comment l'écrêtage du gradient stabilise-t-il l'entraînement ?

In [ ]:
# ✍️ Essaie ici


## Les GRU

Les GRU (Gated Recurrent Units) sont une variante simplifiée des LSTM : même capacité à capter les dépendances longues, mais moins de paramètres, donc plus rapides. Pas de cellule mémoire séparée : l'information est stockée dans l'état caché.

| Porte | Rôle |
| --- | --- |
| Porte de mise à jour (update) | Combien garder de l'état précédent et combien ajouter de nouveau |
| Porte de réinitialisation (reset) | Combien oublier de l'état précédent pour calculer le nouveau |

- 1. Le GRU reçoit l'entrée courante et l'état caché précédent.
- 2. Il calcule les deux portes (valeurs entre 0 et 1).
- 3. Il met à jour l'état caché : la porte reset décide quoi oublier, la porte update quoi ajouter.
- 4. Il produit la sortie du pas et le nouvel état caché.

In [ ]:
class SimpleGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, h_n = self.gru(x)            # out : tous les états cachés
        return self.fc(out[:, -1, :])

model = SimpleGRU(input_size=8, hidden_size=16, num_layers=2, output_size=4)
x = torch.randn(3, 7, 8)                  # batch=3, seq_len=7, features=8
print("Output shape:", model(x).shape)

**Sortie :**

```text
Output shape: torch.Size([3, 4])
```

> ✏️ **À toi**
>
> Tu construis une reconnaissance vocale en temps réel sur un flux audio continu. Pourquoi un GRU serait-il un meilleur choix qu'un LSTM ?

In [ ]:
# ✍️ Essaie ici


## RNN, LSTM ou GRU ?

| Critère | RNN classique | LSTM | GRU |
| --- | --- | --- | --- |
| Portes | Aucune | 3 (entrée, oubli, sortie) | 2 (mise à jour, réinitialisation) |
| Cellule mémoire | Non | Oui | Non (état caché) |
| Complexité | Simple | Complexe | Plus simple qu'un LSTM |
| Coût de calcul | Faible | Élevé | Moyen |
| Séquences longues | Difficile | Excellent | Bon |
| Usage | Séquences courtes | Tâches longues et complexes | Séquences longues avec contrainte d'efficacité |

> ✏️ **À toi**
>
> Pour un modèle de prédiction de texte sur mobile, où la mémoire et les dépendances longues comptent, choisirais-tu un RNN, un LSTM ou un GRU ? Justifie (coût, séquences longues, mémoire).

In [ ]:
# ✍️ Essaie ici


## Applications et limites

Applications : prévision boursière et météo, génération de texte, traduction, analyse de sentiment, reconnaissance d'actions dans une vidéo, génération musicale.

- Coût : plus lourd qu'un RNN, plus léger qu'un LSTM.
- Hyperparamètres : moins nombreux que pour un LSTM, mais à régler.
- Surapprentissage : fréquent sur de petits jeux de données.

> ✏️ **À toi**
>
> Ton GRU d'analyse de sentiment sur un petit jeu d'avis clients excelle à l'entraînement mais échoue en validation. Quelles stratégies contre le surapprentissage ?

In [ ]:
# ✍️ Essaie ici


> ✏️ **À toi**
>
> Tu dois construire un chatbot de service client qui comprend les dépendances longues d'une conversation, répond en temps réel et tient la charge de millions d'utilisateurs. RNN classique, LSTM ou GRU : lequel choisis-tu, et pourquoi ?

In [ ]:
# ✍️ Essaie ici


# 4. Tutoriel : prédire un cours de bourse avec LSTM et GRU

*Les cours d'IBM de 2006 à 2018 avec Keras*

**Objectifs**

- Prétraiter une série temporelle pour un RNN
- Construire et entraîner un LSTM
- Construire et entraîner un GRU
- Comparer leurs performances avec la RMSE

## Environnement et données

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("fivethirtyeight")
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout, GRU
from keras.optimizers import SGD

| Bibliothèque | Rôle |
| --- | --- |
| numpy, pandas | Calcul et manipulation des données |
| matplotlib | Visualiser les tendances |
| MinMaxScaler | Ramener les valeurs entre 0 et 1 pour faciliter la convergence |
| Sequential, Dense, LSTM, GRU, Dropout | Construire le réseau ; Dropout limite le surapprentissage |
| mean_squared_error | Mesurer l'écart entre prédictions et réalité |

In [ ]:
def plot_predictions(test, predicted):
    plt.plot(test, color="red", label="Real IBM Stock Price")
    plt.plot(predicted, color="blue", label="Predicted IBM Stock Price")
    plt.title("IBM Stock Price Prediction")
    plt.xlabel("Time")
    plt.ylabel("IBM Stock Price")
    plt.legend()
    plt.show()

def return_rmse(test, predicted):
    rmse = math.sqrt(mean_squared_error(test, predicted))
    print("The root mean squared error is {}.".format(rmse))

On utilise le prix le plus haut du jour (colonne High) : apprentissage avant 2017, test à partir de 2017.

In [ ]:
dataset = pd.read_csv("IBM_2006-01-01_to_2018-01-01.csv", index_col="Date", parse_dates=["Date"])
training_set = dataset[:"2016"].iloc[:, 1:2].values   # colonne High
test_set = dataset["2017":].iloc[:, 1:2].values

dataset["High"][:"2016"].plot(figsize=(16, 4), legend=True)
dataset["High"]["2017":].plot(figsize=(16, 4), legend=True)
plt.legend(["Training set (Before 2017)", "Test set (2017 and beyond)"])
plt.title("IBM Stock Price")
plt.show()

## Préparer les séquences

In [ ]:
sc = MinMaxScaler(feature_range=(0, 1))
training_set_scaled = sc.fit_transform(training_set)

Pour chaque jour i, le modèle regarde les 60 jours précédents pour prédire le jour i :

In [ ]:
X_train, y_train = [], []
for i in range(60, len(training_set_scaled)):
    X_train.append(training_set_scaled[i-60:i, 0])
    y_train.append(training_set_scaled[i, 0])
X_train, y_train = np.array(X_train), np.array(y_train)

# Format attendu par un RNN : (échantillons, pas de temps, caractéristiques)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

## Le modèle LSTM

In [ ]:
regressor = Sequential()
regressor.add(LSTM(units=50, return_sequences=True, input_shape=(X_train.shape[1], 1)))
regressor.add(Dropout(0.2))
regressor.add(LSTM(units=50, return_sequences=True))
regressor.add(Dropout(0.2))
regressor.add(LSTM(units=50, return_sequences=True))
regressor.add(Dropout(0.2))
regressor.add(LSTM(units=50))
regressor.add(Dropout(0.2))
regressor.add(Dense(units=1))

regressor.compile(optimizer="rmsprop", loss="mean_squared_error")
regressor.fit(X_train, y_train, epochs=50, batch_size=32)

- Quatre couches LSTM de 50 unités ; les trois premières renvoient toute la séquence (return_sequences=True) pour alimenter la suivante, la dernière non.
- Dropout(0.2) désactive 20 % des neurones à l'entraînement.
- Dense(1) produit un prix ; optimiseur RMSprop, perte MSE ; 50 époques par lots de 32.

**Sortie :**

```text
Epoch 1/50 ... loss: 0.0266
Epoch 2/50 ... loss: 0.0094
Epoch 3/50 ... loss: 0.0085
```

## Prédire et évaluer

On reconstitue des fenêtres de 60 jours pour la période de test, en reprenant les 60 derniers jours d'entraînement pour le contexte, et on applique le même scaler :

In [ ]:
dataset_total = pd.concat((dataset["High"][:"2016"], dataset["High"]["2017":]), axis=0)
inputs = dataset_total[len(dataset_total) - len(test_set) - 60:].values
inputs = sc.transform(inputs.reshape(-1, 1))

X_test = []
for i in range(60, len(inputs)):
    X_test.append(inputs[i-60:i, 0])
X_test = np.reshape(np.array(X_test), (len(X_test), 60, 1))

predicted_stock_price = sc.inverse_transform(regressor.predict(X_test))   # retour à l'échelle réelle
plot_predictions(test_set, predicted_stock_price)
return_rmse(test_set, predicted_stock_price)

> 💡 **Pourquoi la RMSE ?** : La RMSE (racine de l'erreur quadratique moyenne) pénalise davantage les grosses erreurs et s'exprime dans l'unité du prix : plus elle est basse, meilleur est le modèle.

## Le modèle GRU

In [ ]:
regressorGRU = Sequential()
regressorGRU.add(GRU(units=50, return_sequences=True, input_shape=(X_train.shape[1], 1), activation="tanh"))
regressorGRU.add(Dropout(0.2))
regressorGRU.add(GRU(units=50, return_sequences=True, activation="tanh"))
regressorGRU.add(Dropout(0.2))
regressorGRU.add(GRU(units=50, return_sequences=True, activation="tanh"))
regressorGRU.add(Dropout(0.2))
regressorGRU.add(GRU(units=50, activation="tanh"))
regressorGRU.add(Dropout(0.2))
regressorGRU.add(Dense(units=1))

regressorGRU.compile(optimizer=SGD(learning_rate=0.01, momentum=0.9, nesterov=False), loss="mean_squared_error")
regressorGRU.fit(X_train, y_train, epochs=50, batch_size=150)

GRU_predicted = sc.inverse_transform(regressorGRU.predict(X_test))
plot_predictions(test_set, GRU_predicted)
return_rmse(test_set, GRU_predicted)

- learning_rate=0.01 : l'ampleur des mises à jour.
- momentum=0.9 : aide l'optimiseur à éviter les minima locaux.
- nesterov=False : sans gradient accéléré de Nesterov.

> ⚠️ **Versions** : Dans les versions récentes de Keras, SGD(lr=…, decay=…) n'existe plus : utilise learning_rate et, si besoin, un planning de taux d'apprentissage.

Ici, le LSTM est légèrement meilleur que le GRU, mais le GRU est plus rapide à entraîner. Le bon choix dépend du jeu de données et de la tâche.

---
# Exercices de la journée
Ouvre chaque notebook d'exercices dans Colab, puis rends-le via GitHub sur Propilote-Learning :

- [Exercises XP : séries temporelles avec un LSTM](https://colab.research.google.com/github/franckkohou/propilote-notebooks/blob/main/semaine-5/jour-4/exercices-xp-series-temporelles-lstm.ipynb)
- [Exercises XP Gold](https://colab.research.google.com/github/franckkohou/propilote-notebooks/blob/main/semaine-5/jour-4/exercices-xp-gold.ipynb)
- [Exercises XP Ninja](https://colab.research.google.com/github/franckkohou/propilote-notebooks/blob/main/semaine-5/jour-4/exercices-xp-ninja.ipynb)
- [Daily Challenge : prédiction boursière avec un LSTM](https://colab.research.google.com/github/franckkohou/propilote-notebooks/blob/main/semaine-5/jour-4/daily-challenge-prediction-boursiere-lstm.ipynb)